In [ ]:
# Celula unica - Importacao de dados GPKG para PostgreSQL
import geopandas as gpd
import pandas as pd
from sqlalchemy import create_engine, text
from pathlib import Path

# =============
# CONFIGURACOES 
# =============

# Pasta onde estao os arquivos GPKG
PASTA_DADOS = Path(r"C:\Temp")

# Lista de arquivos CAR
CAR_ARQUIVOS = [
    "es_pi_car_20260406"
]

# Arquivo SIGEF
SIGEF_ARQUIVO = "pa_br_sigef_privado_20260107"

# Controle de importacao
IMPORTAR_CAR = True
IMPORTAR_SIGEF = False              #ATIVAR COM "True" CASO AINDA NÃO TENHA IMPORTADO A TABELA NA AULA ANTERIOR!

# Se False, mantem a tabela existente no banco (nao reimporta)
RECRIAR_TABELAS = False

# Credenciais do banco local
DB_HOST = "localhost"
DB_PORT = "5432"
DB_USER = "postgres"
DB_PASSWORD = "postgres"
DB_NAME = "geotec"

# ==========================================
# FUNCOES
# ==========================================

def conectar_banco():
    return create_engine(f'postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}')


def criar_schemas(engine):
    with engine.connect() as conn:
        conn.execute(text("CREATE SCHEMA IF NOT EXISTS car"))
        conn.execute(text("CREATE SCHEMA IF NOT EXISTS incra"))
        conn.execute(text("CREATE EXTENSION IF NOT EXISTS postgis"))
        conn.commit()
    print("[OK] Schemas car e incra verificados/criados")


def verificar_tabela(engine, schema, tabela):
    try:
        with engine.connect() as conn:
            conn.execute(text(f"SELECT 1 FROM {schema}.{tabela} LIMIT 1"))
            return True
    except:
        return False


def contar_registros(engine, schema, tabela):
    try:
        with engine.connect() as conn:
            result = conn.execute(text(f"SELECT COUNT(*) FROM {schema}.{tabela}"))
            return result.fetchone()[0]
    except:
        return 0


def encontrar_arquivo(pasta, nome_arquivo):
    """Tenta localizar o arquivo: .gpkg -> .shp"""
    nome_base = Path(nome_arquivo).stem
    
    # Tenta GPKG primeiro
    gpkg_path = pasta / f"{nome_base}.gpkg"
    if gpkg_path.exists():
        return gpkg_path, 'gpkg'
    
    # Tenta SHP
    shp_path = pasta / f"{nome_base}.shp"
    if shp_path.exists():
        return shp_path, 'shp'
    
    return None, None


def importar_arquivo(engine, caminho, tipo_arquivo, schema, tabela):
    """Importa arquivo (GPKG ou SHP) para o PostgreSQL"""
    print(f"  Lendo: {caminho.name} (formato: {tipo_arquivo.upper()})")
    
    try:
        gdf = gpd.read_file(str(caminho))
    except Exception as e:
        print(f"  ERRO ao ler arquivo: {e}")
        return False
    
    print(f"  Registros encontrados: {len(gdf)}")

    tem_geometria = (
        isinstance(gdf, gpd.GeoDataFrame)
        and gdf.active_geometry_name is not None
        and not gdf.geometry.isna().all()
    )

    if tem_geometria:
        if gdf.crs is None:
            gdf = gdf.set_crs(epsg=4674)
        else:
            gdf = gdf.to_crs(epsg=4674)
        gdf.to_postgis(
            name=tabela, con=engine, schema=schema,
            if_exists="replace", index=False
        )
    else:
        df = pd.DataFrame(gdf)
        geom_col = getattr(gdf, "active_geometry_name", None)
        if geom_col and geom_col in df.columns:
            df = df.drop(columns=geom_col)
        print("  [INFO] Sem geometria ativa; importando como tabela comum")
        df.to_sql(
            name=tabela, con=engine, schema=schema,
            if_exists="replace", index=False
        )

    print(f"  Importado para {schema}.{tabela}")
    return True


def importar_se_necessario(engine, nome_arquivo, schema, tabela, ativo):
    """Versão atualizada: tenta GPKG, depois SHP"""
    if not ativo:
        print(f"  [SKIP] {schema}.{tabela} (desativado)")
        return

    caminho, tipo_arquivo = encontrar_arquivo(PASTA_DADOS, nome_arquivo)
    
    if caminho is None:
        print(f"  [ERRO] Arquivo nao encontrado: {nome_arquivo} (GPKG ou SHP) em {PASTA_DADOS}")
        return

    if verificar_tabela(engine, schema, tabela) and not RECRIAR_TABELAS:
        n = contar_registros(engine, schema, tabela)
        print(f"  [EXISTE] {schema}.{tabela} ({n} registros) - mantido")
        return

    importar_arquivo(engine, caminho, tipo_arquivo, schema, tabela)


# ==========================================
# MAIN
# ==========================================

def main():
    print("=" * 50)
    print("IMPORTACAO DE DADOS GEOPROCESSAMENTO")
    print("=" * 50)
    
    # Verificar pasta
    if not PASTA_DADOS.exists():
        print(f"\n[ERRO] Pasta nao encontrada: {PASTA_DADOS}")
        return
    
    arquivos_faltando = []

    # Verificar CARs
    if IMPORTAR_CAR:
        for arquivo in CAR_ARQUIVOS:
            caminho, _ = encontrar_arquivo(PASTA_DADOS, arquivo)
            if caminho is None:
                arquivos_faltando.append(arquivo)

    # Verificar SIGEF
    if IMPORTAR_SIGEF:
        caminho, _ = encontrar_arquivo(PASTA_DADOS, SIGEF_ARQUIVO)
        if caminho is None:
            arquivos_faltando.append(SIGEF_ARQUIVO)

    if arquivos_faltando:
        print(f"\n[ERRO] Arquivos nao encontrados: {arquivos_faltando}")
        return
    
    # Conexao
    print("\n[1] Conectando ao PostgreSQL...")
    try:
        engine = conectar_banco()
        print("[OK] Conexao estabelecida")
    except Exception as e:
        print(f"[ERRO] {e}")
        return
    
    # Schemas
    print("\n[2] Configurando schemas...")
    criar_schemas(engine)
    
    # =====================
    # IMPORTAR CAR
    # =====================
    if IMPORTAR_CAR:
        print("\n[3] Importando CAR...")
        
        for arquivo in CAR_ARQUIVOS:
            tabela = Path(arquivo).stem
            
            print(f"\n--> {arquivo}")
            
            if verificar_tabela(engine, "car", tabela) and not RECRIAR_TABELAS:
                registros = contar_registros(engine, "car", tabela)
                print(f"[INFO] car.{tabela} ja existe ({registros} registros)")
                
                resposta = input("[?] Recriar? (s/N): ")
                if resposta.lower() == 's':
                    importar_se_necessario(engine, arquivo, "car", tabela, True)
                else:
                    print("  Mantido")
            else:
                importar_se_necessario(engine, arquivo, "car", tabela, True)

    else:
        print("\n[3] Importacao do CAR desativada")

    # =====================
    # IMPORTAR SIGEF
    # =====================
    if IMPORTAR_SIGEF:
        print("\n[4] Importando SIGEF...")
        
        tabela = Path(SIGEF_ARQUIVO).stem
        
        if verificar_tabela(engine, "incra", tabela) and not RECRIAR_TABELAS:
            registros = contar_registros(engine, "incra", tabela)
            print(f"[INFO] incra.{tabela} ja existe ({registros} registros)")
            
            resposta = input("[?] Recriar? (s/N): ")
            if resposta.lower() == 's':
                importar_se_necessario(engine, SIGEF_ARQUIVO, "incra", tabela, True)
            else:
                print("  Mantido")
        else:
            importar_se_necessario(engine, SIGEF_ARQUIVO, "incra", tabela, True)

    else:
        print("\n[4] Importacao do SIGEF desativada")

    # =====================
    # RESUMO FINAL
    # =====================
    if IMPORTAR_CAR:
        print("\nResumo CAR:")
        for arquivo in CAR_ARQUIVOS:
            tabela = Path(arquivo).stem
            registros = contar_registros(engine, "car", tabela)
            print(f"  {tabela}: {registros} registros")

    if IMPORTAR_SIGEF:
        tabela = Path(SIGEF_ARQUIVO).stem
        registros = contar_registros(engine, "incra", tabela)
        print(f"\nSIGEF: {registros} registros")

    engine.dispose()
    
    print("\n" + "=" * 50)
    print("PRONTO! AGORA PODEMOS SEGUIR COM O CURSO")
    print("=" * 50)


if __name__ == "__main__":
    main()